# Hybrid RAG — LMaaS-built Knowledge Graph

A clean, linear demo where **real LMaaS creates the knowledge graph** — the *nodes*, their *type labels*,
and the *relationship edges* — by extracting subject-relation-object **triples** from the service-manual docs.

- **LLM (LMaaS):** real — Azure OpenAI via the IDAM two-leg token-exchange flow.
- **Vector store (DIS):** fake (in-memory) — not needed to show LLM graph construction.
- **Graph store:** in-memory (the extracted triples), visualised with vis-network.

Run the cells top-to-bottom.

## 0. Prerequisites

You need your two IDAM secrets: `IDAM_APP_CLIENT_ID` and `IDAM_APP_CLIENT_SECRET`.

**On SageMaker**, running `export VAR=...` in a Terminal does *not* reach this notebook's kernel (kernels
inherit the Jupyter server's environment, not your shell's). So the credentials cell in section 1 **prompts
you for them** with `getpass` (masked input) — the values are set for this kernel only and are **never saved
into the notebook**. Just run the cell and paste each value when asked.

If you'd rather set them ahead of time (e.g. from a SageMaker Lifecycle Config or `%env`), do that and the
cell will pick them up from `os.environ` without prompting.

Also: upload the whole `hybrid-rag/` folder — this notebook imports the `hybridrag` package and reads `../docs`.

In [ ]:
%pip install -q -r ../requirements.txt
import importlib.util
print("openai present:  ", importlib.util.find_spec("openai") is not None)
print("langgraph present:", importlib.util.find_spec("langgraph") is not None)
print("(if False, or a later import fails, restart the kernel and re-run)")

## 1. Connect to real LMaaS

Fill the LMaaS config (values from the aifabric-lmaas sample), build the client, and make one tiny call to
confirm the token-exchange + gateway path works end-to-end.

In [ ]:
import os, sys, tempfile
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))  # make hybridrag importable

from hybridrag import config
# From SageMaker/AWS use the Internal VPCE DNS (via PrivateLink). The External host
# (lmaas-integ-int...) is corporate-network only and returns an awselb 403 from AWS.
config.LMAAS_ENDPOINT          = "https://lmaas-integ.ailab.gehealthcare.net"        # Internal VPCE DNS (SageMaker)
# config.LMAAS_ENDPOINT        = "https://lmaas-integ-int.ailab.gehealthcare.net"    # External (corp network / VPN)
config.LMAAS_AUDIENCE          = "0_b2dJB20TBhxzLIHCMzSG4RiQYa"
config.LMAAS_API_VERSION       = "2025-04-01-preview"
config.LMAAS_DEPLOYMENT_STRONG = "integ-gpt-5.2-2025-12-11"   # used for triple extraction (quality)
config.LMAAS_DEPLOYMENT_CHEAP  = "integ-gpt-5.2-2025-12-11"   # swap for a mini deployment if you have one
config.IDAM_TOKEN_ENDPOINT     = "https://idam.gehealthcloud.io/oauth2/token"
# config.LMAAS_VERIFY_SSL = False   # uncomment if you hit SSLCertVerificationError on the GE network

import getpass
# Uses env vars if already set; otherwise prompts (masked) — secrets are NOT saved in the notebook.
config.IDAM_APP_CLIENT_ID     = os.environ.get("IDAM_APP_CLIENT_ID")     or getpass.getpass("IDAM_APP_CLIENT_ID: ")
config.IDAM_APP_CLIENT_SECRET = os.environ.get("IDAM_APP_CLIENT_SECRET") or getpass.getpass("IDAM_APP_CLIENT_SECRET: ")

from hybridrag.clients import LMaaSClient
lmaas = LMaaSClient()
print("health check ->", lmaas.complete_text("You are a health check.", "Reply with the single word OK."))

## 2. Load the documents

In [ ]:
from hybridrag.loader import load_documents
docs = load_documents(os.path.join("..", "docs"))
for d in docs:
    print(f"  {d['doc_id']:30s} {len(d['text'].split()):4d} words   {d['title']}")

## 3. Let LMaaS build the knowledge graph

`KGExtractor` windows each document and asks LMaaS for **typed subject-relation-object triples**. Entities
merge by name across windows and documents, so repeated names collapse into one shared node — that is what
turns a list of triples into a connected graph. Each entity gets an **LLM-assigned type label** and each
triple becomes a **typed relationship edge**.

In [ ]:
from hybridrag.kg_extract import KGExtractor

extractor = KGExtractor(lmaas, cheap=False)   # strong deployment for extraction quality
kg = extractor.build(docs, on_doc=lambda doc_id, n: print(f"  {doc_id:30s} +{n} triples"))

print(f"\nKnowledge graph: {len(kg.nodes)} nodes, {len(kg.edges)} edges")
print("entity types (label -> count):")
for t, c in kg.type_counts().items():
    print(f"    {t:14s} {c}")

## 4. Inspect the extracted triples

The raw graph facts LMaaS produced — subject `[type]` -> **RELATION** -> object `[type]`.

In [ ]:
triples = kg.triples()
print(f"{len(triples)} distinct triples (showing up to 30):\n")
for t in triples[:30]:
    print(f"  {t['subject']} [{t['subject_type']}]  --{t['relation']}-->  {t['object']} [{t['object_type']}]")

## 5. Visualise the LMaaS-built knowledge graph

Force-directed view: nodes **coloured by their LLM-assigned type**, edges **labelled with the relation**.
Hover a node for its type and how many documents mention it.

In [ ]:
from IPython.display import HTML
from hybridrag import visualize

nodes, edges = kg.to_vis()
HTML(visualize.iframe_srcdoc(
    visualize.entity_kg_html(nodes, edges, "LMaaS-built Knowledge Graph — service manuals"), 620))

## 6. Let LMaaS build the PageIndex tree

The **PageIndex** is the table-of-contents-like hierarchy the *vectorless* retriever navigates to find where
an answer lives. Here LMaaS reads each document and returns the nested outline directly — chapters and
sub-sections with one-line summaries — so the **nesting is LLM-built**, not inferred from heading numbers.

In [ ]:
from hybridrag.kg_extract import PageIndexExtractor

pidx = PageIndexExtractor(lmaas, cheap=False)
trees = pidx.build(docs, on_doc=lambda doc_id, n: print(f"  {doc_id:30s} {n} sections"))

In [ ]:
DOC = "imaging_service_manual"   # change to any doc_id printed above
HTML(visualize.iframe_srcdoc(
    visualize.pageindex_tree_html(trees[DOC], f"LMaaS-built PageIndex — {DOC}"), 620))

## 7. Connect the real graph store (Amazon Neptune)

Sections 3–6 held everything in memory. Now we point Agent 1 and Agent 2 at a **real Neptune cluster**, so the
document tree is *persisted* and the CRAG loop retrieves from it. **DIS stays fake** (in-memory vector store).

Fill in your cluster endpoint below, then the connectivity check must pass before continuing.

> Neptune reachability is separate from LMaaS: the notebook must be in (or peered to) the **Neptune VPC**, and
> with `NEPTUNE_USE_IAM = True` the notebook's IAM role needs Neptune data-access permission (requests are
> SigV4-signed via boto3).

In [ ]:
config.NEPTUNE_ENDPOINT = "REPLACE_ME"   # <-- cluster WRITER endpoint, no https://, no :8182
config.NEPTUNE_PORT     = 8182
config.NEPTUNE_USE_IAM  = True            # False if IAM DB auth is disabled on the cluster
config.AWS_REGION       = "us-east-1"     # <-- your Neptune region

assert config.NEPTUNE_ENDPOINT != "REPLACE_ME", "Set config.NEPTUNE_ENDPOINT above first!"

from hybridrag.clients import GraphClient
neptune = GraphClient()
print("connectivity:", neptune.run_gremlin("g.V().limit(1).count()"))

### 7.1 Persist the LMaaS-built knowledge graph into Neptune

Sections 3–5 built the typed entity graph **in memory**. This writes it to Neptune: each entity becomes a
vertex **labelled with its LLM-assigned type** (and carrying an `etype` property), and each extracted triple
becomes a **typed edge** (`RESOLVED_BY`, `POSITIONS`, …) with its occurrence `count`.

Writes are **idempotent** — vertices upsert by id, edges upsert on (subject, relation, object) — so re-running
this cell never duplicates the graph. We then read it **back out of Neptune** and re-render it, which proves the
round-trip rather than just re-showing the in-memory object.

In [ ]:
kg_nodes, kg_edges = kg.to_store()
print(f"writing {len(kg_nodes)} entities + {len(kg_edges)} relationships to Neptune ...")
neptune.upsert_typed_entities(kg_nodes, kg_edges)
print("done")

# read the graph BACK from Neptune (not from memory) and render it
live_nodes, live_edges = neptune.get_entity_graph()
print(f"read back from Neptune: {len(live_nodes)} entities, {len(live_edges)} relationships")
HTML(visualize.iframe_srcdoc(
    visualize.entity_kg_html(live_nodes, live_edges,
                             "LIVE Neptune — LMaaS-built knowledge graph"), 620))

### 7.2 Ingest the document tree into Neptune, then query it

The **same real LMaaS** drives Agent 1 (ingest routing — the entity gate) and Agent 2 (the CRAG query loop).
Agent 1 now writes the `Document → Section → Chunk` tree plus `Section → MENTIONS → Entity` edges into Neptune.

> **This is the slow cell.** Writes go out as individual Gremlin statements (one HTTP POST each), so a document
> with many sections/chunks can take a while. Start with one document (`docs[:1]`) to validate the path, then
> run all of them.

In [ ]:
from hybridrag.fakes import FakeDIS
from hybridrag.catalog import RoutingCatalog
from hybridrag.ingest import IngestAgent
from hybridrag.crag import CRAGQueryAgent

cat_path = os.path.join(tempfile.gettempdir(), "lmaas_kg_neptune_catalog.json")
if os.path.exists(cat_path):
    os.remove(cat_path)
catalog = RoutingCatalog(path=cat_path)

dis, graph = FakeDIS(), neptune          # real graph store; DIS still fake
INGEST_DOCS = docs                       # <-- use docs[:1] for a fast first pass

ingest = IngestAgent(lmaas, dis, graph, catalog=catalog, collection="demo", min_graph_entities=5)
query  = CRAGQueryAgent(lmaas, dis, graph, catalog=catalog, collection="demo")
for e in ingest.ingest_documents(INGEST_DOCS):
    print(f"  {e.doc_id:30s} graph={e.in_graph} vector={e.in_vector} entities={e.meta['entity_count']}")

In [ ]:
# confirm it really landed in Neptune (read back the persisted tree)
for d in INGEST_DOCS:
    secs = neptune.get_sections([d["doc_id"]])
    if secs:
        print(f"{d['doc_id']:30s} {len(secs):3d} sections in Neptune")

DOC = INGEST_DOCS[0]["doc_id"]
live_tree = neptune.get_tree([DOC])
HTML(visualize.iframe_srcdoc(
    visualize.graph_html(live_tree, f"LIVE Neptune — {DOC}"), 620))

In [ ]:
def ask(q):
    r = query.run(q)
    print(f"Q: {q}")
    print(f"   verdict : {r.verdict.label} (score={r.verdict.score:.2f})")
    print(f"   sources : {r.sources}   LLM calls: {r.llm_calls}")
    print(f"   answer  : {r.answer[:220]}\n")

ask("What does error code E-204 mean and how do I fix it?")
ask("Which component does the Gantry position?")

### 7.3 Clean up (optional)

Every write in this notebook is idempotent, so you don't *need* to clean up before re-running. Uncomment to
remove the demo data from Neptune — the document tree and/or the LMaaS entity graph.

In [ ]:
# # document tree (Document / Section / Chunk)
# for d in INGEST_DOCS:
#     did = d["doc_id"]
#     neptune.run_gremlin(f"g.V().has('docId','doc_{did}').drop()")   # sections + chunks
#     neptune.run_gremlin(f"g.V('doc_{did}').drop()")                 # the Document node
#
# # LMaaS-built typed entity graph (everything carrying an 'etype' property)
# neptune.run_gremlin("g.V().has('etype').drop()")
# print("cleaned up")

---
**Recap:** real LMaaS built both structures — the typed **knowledge graph** (nodes + type labels +
relationship edges, sections 3–5) and the **PageIndex tree** (LLM-built nesting, section 6). Section 7 then put
**all graph building on Neptune**: the LMaaS entity graph is persisted as typed vertices/edges (§7.1, read back
live to prove the round-trip), the document tree is written by Agent 1 (§7.2), and the CRAG loop retrieves from
Neptune. The only remaining stand-in is **DIS** (in-memory vector store) — swap `FakeDIS` for `DISClient` in
`hybridrag.clients` once the DIS API paths are configured to go fully live.